In [ ]:
# Cell 1: imports, config, constants
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

FILE_PATH  = 'fichinH125.xlsx'   # ← set path here
B          = 2000
SEED       = 42
DROP_R     = False               # match diagnostic notebook setting

SESSION_START  = pd.Timedelta('9:00:00')
SESSION_END    = pd.Timedelta('13:30:00')
BIN_MINUTES    = 15
N_BINS         = 18
Y_MAX          = 0.12

BIN_LABEL_STRS = [f'{9 + 15*i//60:02d}:{15*i%60:02d}' for i in range(N_BINS)]
WEEKDAY_ORDER  = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes']
WEEKDAY_COLORS = {
    'lunes':     '#1f77b4',
    'martes':    '#ff7f0e',
    'miércoles': '#2ca02c',
    'jueves':    '#d62728',
    'viernes':   '#9467bd',
}

In [ ]:
# Cell 2: pure functions

def load_data(path):
    df = pd.read_excel(path)
    if df['Fecha'].dtype == 'object':
        df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True)
    else:
        df['Fecha'] = pd.to_datetime(df['Fecha']).dt.normalize()
    def to_td(t):
        if isinstance(t, str):
            return pd.to_timedelta(t)
        if hasattr(t, 'hour'):
            return pd.Timedelta(hours=t.hour, minutes=t.minute, seconds=t.second)
        return pd.to_timedelta(t)
    df['Tiempo_td'] = df['Tiempo'].apply(to_td)
    return df
def assign_bin(tiempo_td):
    # -1 outside [9:00, 13:30)
    if tiempo_td < SESSION_START or tiempo_td >= SESSION_END:
        return -1
    return int((tiempo_td - SESSION_START).total_seconds() // 60) // BIN_MINUTES
def session_shares(df, drop_r=False):
    # returns DataFrame: Fecha, bin_idx, share, dow
    wk = df[df['Reg'] != 'R'].copy() if drop_r else df.copy()
    wk['bin_idx'] = wk['Tiempo_td'].apply(assign_bin)
    in_sess = wk[wk['bin_idx'] >= 0]
    bv = in_sess.groupby(['Fecha', 'bin_idx'])['Monto'].sum().reset_index()
    idx = pd.MultiIndex.from_product(
        [wk['Fecha'].unique(), range(N_BINS)], names=['Fecha', 'bin_idx']
    )
    bv = bv.set_index(['Fecha', 'bin_idx']).reindex(idx, fill_value=0).reset_index()
    V = bv.groupby('Fecha')['Monto'].transform('sum')
    bv['share'] = np.where(V > 0, bv['Monto'] / V, 0.0)
    bv['dow'] = bv['Fecha'].apply(
        lambda d: ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', None, None][d.weekday()]
    )
    return bv
def bootstrap_ci(share_matrix, B, rng):
    # share_matrix: (N_wd, 18) ndarray — rows are sessions, cols are bins
    N = share_matrix.shape[0]
    boot = np.array([
        share_matrix[rng.integers(0, N, size=N)].mean(axis=0)
        for _ in range(B)
    ])
    return np.percentile(boot, 2.5, axis=0), np.percentile(boot, 97.5, axis=0)
def weekday_plot(wd_results, pooled_mu, bin_labels):
    # wd_results: {wd: {'mu': arr, 'lo': arr, 'hi': arr, 'N': int}}
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=True)
    axes_flat = axes.flatten()
    x = np.arange(len(bin_labels))
    for i, wd in enumerate(WEEKDAY_ORDER):
        ax  = axes_flat[i]
        res = wd_results[wd]
        c   = WEEKDAY_COLORS[wd]
        ax.fill_between(x, res['lo'], res['hi'], alpha=0.25, color=c)
        ax.plot(x, res['mu'], '-o', color=c, lw=1.5, ms=4)
        ax.plot(x, pooled_mu, '--', color='black', lw=0.9)
        ax.set_xticks(x)
        ax.set_xticklabels(bin_labels, rotation=45, ha='right', fontsize=7)
        ax.set_ylim(0, Y_MAX)
        ax.set_title(f'{wd}  (N={res["N"]})', fontsize=9)
        if i % 3 == 0:
            ax.set_ylabel('Share of daily volume')
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
    # panel 6: legend
    ax6 = axes_flat[5]
    ax6.axis('off')
    ax6.plot([], [], '-o', color='grey', lw=1.5, ms=4, label='μ̂_wd(t)  weekday mean')
    ax6.fill_between([], [], [], alpha=0.35, color='grey', label='95% bootstrap CI  (B=2000)')
    ax6.plot([], [], '--', color='black', lw=0.9, label='μ̂_pooled(t)  reference')
    ax6.legend(loc='center', fontsize=9, frameon=False)
    fig.suptitle(
        'Weekday-specific μ(t) vs pooled μ(t) — 95% bootstrap CI',
        fontsize=11, y=1.005
    )
    plt.tight_layout()
    return fig

In [ ]:
# Cell 3: execution

raw = load_data(FILE_PATH)
print(f'Loaded {len(raw)} rows  |  sessions: {raw["Fecha"].nunique()}')

shares = session_shares(raw, drop_r=DROP_R)
print(f'DROP_R={DROP_R}  |  sessions in shares: {shares["Fecha"].nunique()}')

# pooled μ over all sessions
pooled_mu = shares.groupby('bin_idx')['share'].mean().reindex(range(N_BINS)).values

# per-weekday bootstrap
rng = np.random.default_rng(seed=SEED)
wd_results = {}
for wd in WEEKDAY_ORDER:
    sub = shares[shares['dow'] == wd]
    mat = (
        sub.pivot(index='Fecha', columns='bin_idx', values='share')
           .reindex(columns=range(N_BINS), fill_value=0)
           .values
    )
    lo, hi = bootstrap_ci(mat, B, rng)
    wd_results[wd] = {'mu': mat.mean(axis=0), 'lo': lo, 'hi': hi, 'N': mat.shape[0]}
    print(f'  {wd}: N={mat.shape[0]}')

fig = weekday_plot(wd_results, pooled_mu, BIN_LABEL_STRS)
plt.savefig('weekday_bootstrap.png', dpi=120, bbox_inches='tight')
plt.show()

# out-of-CI table
print('\nBins where pooled μ(t) lies outside weekday 95% CI:')
for wd in WEEKDAY_ORDER:
    res = wd_results[wd]
    outside = [
        BIN_LABEL_STRS[t]
        for t in range(N_BINS)
        if pooled_mu[t] < res['lo'][t] or pooled_mu[t] > res['hi'][t]
    ]
    if outside:
        print(f'{wd} (N={res["N"]}): bins outside pooled CI: {outside}')
    else:
        print(f'{wd} (N={res["N"]}): pooled μ within CI at all bins — weekday conditioning not justified')